**ROAD SEVERITY DATA SET PREDICTION**

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [2]:
df=pd.read_csv("/content/Road Accident Data.csv")

**BASIC DATA INSPECTION**

In [3]:
df.head()

,Accident_Index,Accident Date,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,Local_Authority_(District),Carriageway_Hazards,...,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type
0,BS0000001,01-01-2021,Thursday,Give way or uncontrolled,T or staggered junction,Serious,51.512273,Daylight,Kensington and Chelsea,NaN,...,1,2,Metropolitan Police,Dry,One way street,30,15:11,Urban,Fine no high winds,Car
1,BS0000002,05-01-2021,Monday,Give way or uncontrolled,Crossroads,Serious,51.514399,Daylight,Kensington and Chelsea,NaN,...,11,2,Metropolitan Police,Wet or damp,Single carriageway,30,10:59,Urban,Fine no high winds,Taxi/Private hire car
2,BS0000003,04-01-2021,Sunday,Give way or uncontrolled,T or staggered junction,Slight,51.486668,Daylight,Kensington and Chelsea,NaN,...,1,2,Metropolitan Police,Dry,Single carriageway,30,14:19,Urban,Fine no high winds,Taxi/Private hire car
3,BS0000004,05-01-2021,Monday,Auto traffic signal,T or staggered junction,Serious,51.507804,Daylight,Kensington and Chelsea,NaN,...,1,2,Metropolitan Police,Frost or ice,Single carriageway,30,08:10,Urban,Other,Motorcycle over 500cc
4,BS0000005,06-01-2021,Tuesday,Auto traffic signal,Crossroads,Serious,51.482076,Darkness - lights lit,Kensington and Chelsea,NaN,...,1,2,Metropolitan Police,Dry,Single carriageway,30,17:25,Urban,Fine no high winds,Car


In [4]:
# TARGET COLUMN : ACCIDENT SEVERITY
df["Accident_Severity"].value_counts()

,count
Accident_Severity,
Slight,263280
Serious,40740
Fatal,3904
Fetal,49


In [5]:
df.shape

(307973, 21)

In [6]:
df.isnull().sum()

,0
Accident_Index,0
Accident Date,0
Day_of_Week,0
Junction_Control,0
Junction_Detail,0
Accident_Severity,0
Latitude,0
Light_Conditions,0
Local_Authority_(District),0
Carriageway_Hazards,302549


In [7]:
df.drop("Carriageway_Hazards",inplace=True,axis=1)

In [8]:
df.head(
)

,Accident_Index,Accident Date,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,Local_Authority_(District),Longitude,Number_of_Casualties,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Time,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type
0,BS0000001,01-01-2021,Thursday,Give way or uncontrolled,T or staggered junction,Serious,51.512273,Daylight,Kensington and Chelsea,-0.201349,1,2,Metropolitan Police,Dry,One way street,30,15:11,Urban,Fine no high winds,Car
1,BS0000002,05-01-2021,Monday,Give way or uncontrolled,Crossroads,Serious,51.514399,Daylight,Kensington and Chelsea,-0.199248,11,2,Metropolitan Police,Wet or damp,Single carriageway,30,10:59,Urban,Fine no high winds,Taxi/Private hire car
2,BS0000003,04-01-2021,Sunday,Give way or uncontrolled,T or staggered junction,Slight,51.486668,Daylight,Kensington and Chelsea,-0.179599,1,2,Metropolitan Police,Dry,Single carriageway,30,14:19,Urban,Fine no high winds,Taxi/Private hire car
3,BS0000004,05-01-2021,Monday,Auto traffic signal,T or staggered junction,Serious,51.507804,Daylight,Kensington and Chelsea,-0.203110,1,2,Metropolitan Police,Frost or ice,Single carriageway,30,08:10,Urban,Other,Motorcycle over 500cc
4,BS0000005,06-01-2021,Tuesday,Auto traffic signal,Crossroads,Serious,51.482076,Darkness - lights lit,Kensington and Chelsea,-0.173445,1,2,Metropolitan Police,Dry,Single carriageway,30,17:25,Urban,Fine no high winds,Car


In [9]:
# replacing fatel with fatal

# Standardize text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Correct a likely spelling variant
df["Accident_Severity"] = df["Accident_Severity"].replace({
    "Fetal": "Fatal"
})


In [10]:
# TARGET COLUMN : ACCIDENT SEVERITY
df["Accident_Severity"].value_counts()


,count
Accident_Severity,
Slight,263280
Serious,40740
Fatal,3953


In [11]:

# Duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())


Duplicate rows: 0


In [12]:

# Convert date into datetime format
date = pd.to_datetime(
    df["Accident Date"],
    dayfirst=True,
    errors="coerce"
)

df["Accident_Year"] = date.dt.year
df["Accident_Month"] = date.dt.month

# Extract hour from time
time = pd.to_datetime(
    df["Time"],
    format="%H:%M",
    errors="coerce"
)

df["Accident_Hour"] = time.dt.hour

# Remove columns that are not needed in the first model
df.drop(
    columns=[
        "Accident_Index",
        "Accident Date",
        "Time",

        "Number_of_Casualties"
    ],
    inplace=True
)

print("Shape after cleaning:", df.shape)
display(df.head())


Shape after cleaning: (307973, 19)


,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,Local_Authority_(District),Longitude,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Accident_Year,Accident_Month,Accident_Hour
0,Thursday,Give way or uncontrolled,T or staggered junction,Serious,51.512273,Daylight,Kensington and Chelsea,-0.201349,2,Metropolitan Police,Dry,One way street,30,Urban,Fine no high winds,Car,2021,1,15.0
1,Monday,Give way or uncontrolled,Crossroads,Serious,51.514399,Daylight,Kensington and Chelsea,-0.199248,2,Metropolitan Police,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,10.0
2,Sunday,Give way or uncontrolled,T or staggered junction,Slight,51.486668,Daylight,Kensington and Chelsea,-0.179599,2,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,14.0
3,Monday,Auto traffic signal,T or staggered junction,Serious,51.507804,Daylight,Kensington and Chelsea,-0.203110,2,Metropolitan Police,Frost or ice,Single carriageway,30,Urban,Other,Motorcycle over 500cc,2021,1,8.0
4,Tuesday,Auto traffic signal,Crossroads,Serious,51.482076,Darkness - lights lit,Kensington and Chelsea,-0.173445,2,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,2021,1,17.0


In [13]:
df["Accident_Month"].unique()

array([ 1,  2,  3,  4,  5,  6,  8,  7,  9, 10, 11, 12], dtype=int32)

In [14]:
df.isnull().sum()

,0
Day_of_Week,0
Junction_Control,0
Junction_Detail,0
Accident_Severity,0
Latitude,0
Light_Conditions,0
Local_Authority_(District),0
Longitude,0
Number_of_Vehicles,0
Police_Force,0


In [15]:
df["Junction_Control"].unique()

array(['Give way or uncontrolled', 'Auto traffic signal',
       'Data missing or out of range', 'Authorised person', 'Stop sign',
       'Not at junction or within 20 metres', 'Auto traffic sigl'],
      dtype=object)

In [17]:
df["Junction_Control"].value_counts()

,count
Junction_Control,
Give way or uncontrolled,150045
Data missing or out of range,98056
Auto traffic signal,32256
Not at junction or within 20 metres,25378
Stop sign,1685
Authorised person,460
Auto traffic sigl,93


In [18]:

# Check missing values in Accident_Hour
print("Missing hours before filling:",
      df["Accident_Hour"].isnull().sum())

# Calculate the median hour using available values
median_hour = df["Accident_Hour"].median()

# Fill missing values with the median hour
df["Accident_Hour"] = df["Accident_Hour"].fillna(
    round(median_hour)
)

print("Median hour used:", round(median_hour))
print("Missing hours after filling:",
      df["Accident_Hour"].isnull().sum())


Missing hours before filling: 17
Median hour used: 14
Missing hours after filling: 0


In [19]:
df.isnull().sum()

,0
Day_of_Week,0
Junction_Control,0
Junction_Detail,0
Accident_Severity,0
Latitude,0
Light_Conditions,0
Local_Authority_(District),0
Longitude,0
Number_of_Vehicles,0
Police_Force,0


In [20]:
# Standardize text values
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

# Correct a likely spelling variant
df["Junction_Control"] = df["Junction_Control"].replace({
    "Auto traffic sigl": "Auto traffic signal"
})


In [21]:
df["Junction_Control"].value_counts()

,count
Junction_Control,
Give way or uncontrolled,150045
Data missing or out of range,98056
Auto traffic signal,32349
Not at junction or within 20 metres,25378
Stop sign,1685
Authorised person,460


In [22]:
df.head()

,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,Local_Authority_(District),Longitude,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Accident_Year,Accident_Month,Accident_Hour
0,Thursday,Give way or uncontrolled,T or staggered junction,Serious,51.512273,Daylight,Kensington and Chelsea,-0.201349,2,Metropolitan Police,Dry,One way street,30,Urban,Fine no high winds,Car,2021,1,15.0
1,Monday,Give way or uncontrolled,Crossroads,Serious,51.514399,Daylight,Kensington and Chelsea,-0.199248,2,Metropolitan Police,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,10.0
2,Sunday,Give way or uncontrolled,T or staggered junction,Slight,51.486668,Daylight,Kensington and Chelsea,-0.179599,2,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,14.0
3,Monday,Auto traffic signal,T or staggered junction,Serious,51.507804,Daylight,Kensington and Chelsea,-0.203110,2,Metropolitan Police,Frost or ice,Single carriageway,30,Urban,Other,Motorcycle over 500cc,2021,1,8.0
4,Tuesday,Auto traffic signal,Crossroads,Serious,51.482076,Darkness - lights lit,Kensington and Chelsea,-0.173445,2,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Car,2021,1,17.0


In [23]:
# Check unique values in every column
for col in df.columns:
    print(f"\nColumn: {col}")
    print(df[col].unique())


Column: Day_of_Week
['Thursday' 'Monday' 'Sunday' 'Tuesday' 'Friday' 'Wednesday' 'Saturday']

Column: Junction_Control
['Give way or uncontrolled' 'Auto traffic signal'
 'Data missing or out of range' 'Authorised person' 'Stop sign'
 'Not at junction or within 20 metres']

Column: Junction_Detail
['T or staggered junction' 'Crossroads'
 'Not at junction or within 20 metres' 'Roundabout' 'Mini-roundabout'
 'More than 4 arms (not roundabout)' 'Private drive or entrance'
 'Slip road' 'Other junction']

Column: Accident_Severity
['Serious' 'Slight' 'Fatal']

Column: Latitude
[51.512273 51.514399 51.486668 ... 57.585044 57.214898 57.57521 ]

Column: Light_Conditions
['Daylight' 'Darkness - lights lit' 'Darkness - lighting unknown'
 'Darkness - lights unlit' 'Darkness - no lighting']

Column: Local_Authority_(District)
['Kensington and Chelsea' 'Hammersmith and Fulham' 'Westminster'
 'Hounslow' 'City of London' 'Tower Hamlets' 'Hackney' 'Camden'
 'Southwark' 'Brent' 'Haringey' 'Islington' '

In [24]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")

Day_of_Week: 7 unique values
Junction_Control: 6 unique values
Junction_Detail: 9 unique values
Accident_Severity: 3 unique values
Latitude: 264362 unique values
Light_Conditions: 5 unique values
Local_Authority_(District): 422 unique values
Longitude: 269856 unique values
Number_of_Vehicles: 17 unique values
Police_Force: 51 unique values
Road_Surface_Conditions: 5 unique values
Road_Type: 5 unique values
Speed_limit: 8 unique values
Urban_or_Rural_Area: 2 unique values
Weather_Conditions: 8 unique values
Vehicle_Type: 15 unique values
Accident_Year: 2 unique values
Accident_Month: 12 unique values
Accident_Hour: 24 unique values


In [ ]:
df.head(3)

,Day_of_Week,Junction_Control,Junction_Detail,Accident_Severity,Latitude,Light_Conditions,Local_Authority_(District),Longitude,Number_of_Vehicles,Police_Force,Road_Surface_Conditions,Road_Type,Speed_limit,Urban_or_Rural_Area,Weather_Conditions,Vehicle_Type,Accident_Year,Accident_Month,Accident_Hour
0,Thursday,Give way or uncontrolled,T or staggered junction,Serious,51.512273,Daylight,Kensington and Chelsea,-0.201349,2,Metropolitan Police,Dry,One way street,30,Urban,Fine no high winds,Car,2021,1,15.0
1,Monday,Give way or uncontrolled,Crossroads,Serious,51.514399,Daylight,Kensington and Chelsea,-0.199248,2,Metropolitan Police,Wet or damp,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,10.0
2,Sunday,Give way or uncontrolled,T or staggered junction,Slight,51.486668,Daylight,Kensington and Chelsea,-0.179599,2,Metropolitan Police,Dry,Single carriageway,30,Urban,Fine no high winds,Taxi/Private hire car,2021,1,14.0


In [ ]:
# HANDILING THE CATEGORICAL VALUES

In [25]:
# -------------------------------------------------------------------
# STEP 1: Load Data & Clean Up Category Typos/Duplicates
# -------------------------------------------------------------------
# Fix known spelling/duplicate errors found in unique values
df['Local_Authority_(District)'] = df['Local_Authority_(District)'].replace({
    'Stevege': 'Stevenage',
    'Blaeu Gwent': 'Blaenau Gwent',
    'Clackmannshire': 'Clackmannanshire',
    'North Larkshire': 'North Lanarkshire',
    'South Larkshire': 'South Lanarkshire',
    'Crewe and ntwich': 'Crewe and Nantwich'
})

In [26]:
# -------------------------------------------------------------------
# STEP 2: Map Ordinal Categories (Accident_Severity)
# -------------------------------------------------------------------
severity_map = {'Slight': 0, 'Serious': 1, 'Fatal': 2}
df['Accident_Severity_Encoded'] = df['Accident_Severity'].map(severity_map)

In [27]:
# -------------------------------------------------------------------
# STEP 3: Simplify & Group Mixed Text (Vehicle_Type)
# -------------------------------------------------------------------
def simplify_vehicle_type(val):
    if 'Motorcycle' in str(val):
        return 'Motorcycle'
    elif 'Goods' in str(val) or 'Van' in str(val):
        return 'Goods_Vehicle'
    elif 'Bus' in str(val) or 'Minibus' in str(val):
        return 'Bus'
    elif 'Car' in str(val) or 'Taxi' in str(val):
        return 'Car_Taxi'
    else:
        return 'Other'

df['Vehicle_Type_Grouped'] = df['Vehicle_Type'].apply(simplify_vehicle_type)

In [28]:
# -------------------------------------------------------------------
# STEP 4: Frequency Encoding for High-Cardinality Columns
# -------------------------------------------------------------------
# Local Authority has ~300 unique text values
district_counts = df['Local_Authority_(District)'].value_counts()
df['District_Freq'] = df['Local_Authority_(District)'].map(district_counts)

police_counts = df['Police_Force'].value_counts()
df['Police_Force_Freq'] = df['Police_Force'].map(police_counts)

In [29]:
# -------------------------------------------------------------------
# STEP 5: One-Hot Encode Low-Cardinality Nominal Columns
# -------------------------------------------------------------------
nominal_cols = [
    'Day_of_Week', 'Road_Type', 'Road_Surface_Conditions',
    'Weather_Conditions', 'Light_Conditions', 'Urban_or_Rural_Area',
    'Junction_Control', 'Junction_Detail', 'Vehicle_Type_Grouped'
]

# Create dummy variables (1/0)
df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

In [30]:
# -------------------------------------------------------------------
# STEP 6: Define Features (X) and Target (y)
# -------------------------------------------------------------------
# Drop original raw text and unneeded target columns
drop_cols = [
    'Accident_Severity', 'Accident_Severity_Encoded',
    'Local_Authority_(District)', 'Police_Force', 'Vehicle_Type'
]

X = df_encoded.drop(columns=drop_cols)
y = df_encoded['Accident_Severity_Encoded']

print("Features Shape:", X.shape)
print("Target Distribution:\n", y.value_counts())

Features Shape: (307973, 52)
Target Distribution:
 Accident_Severity_Encoded
0    263280
1     40740
2      3953
Name: count, dtype: int64


In [ ]:
X

,Latitude,Longitude,Number_of_Vehicles,Speed_limit,Accident_Year,Accident_Month,Accident_Hour,District_Freq,Police_Force_Freq,Day_of_Week_Monday,...,Junction_Detail_Not at junction or within 20 metres,Junction_Detail_Other junction,Junction_Detail_Private drive or entrance,Junction_Detail_Roundabout,Junction_Detail_Slip road,Junction_Detail_T or staggered junction,Vehicle_Type_Grouped_Car_Taxi,Vehicle_Type_Grouped_Goods_Vehicle,Vehicle_Type_Grouped_Motorcycle,Vehicle_Type_Grouped_Other
0,51.512273,-0.201349,2,30,2021,1,15.0,1376,46789,False,...,False,False,False,False,False,True,True,False,False,False
1,51.514399,-0.199248,2,30,2021,1,10.0,1376,46789,True,...,False,False,False,False,False,False,True,False,False,False
2,51.486668,-0.179599,2,30,2021,1,14.0,1376,46789,False,...,False,False,False,False,False,True,True,False,False,False
3,51.507804,-0.203110,2,30,2021,1,8.0,1376,46789,True,...,False,False,False,False,False,True,False,False,True,False
4,51.482076,-0.173445,2,30,2021,1,17.0,1376,46789,False,...,False,False,False,False,False,False,True,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
307968,57.374005,-3.467828,1,60,2022,2,7.0,809,1009,False,...,True,False,False,False,False,False,True,False,False,False
307969,57.232273,-3.809281,1,60,2022,2,3.0,809,1009,False,...,True,False,False,False,False,False,True,False,False,False
307970,57.585044,-3.862727,3,30,2022,2,9.0,809,1009,False,...,False,False,False,False,False,True,True,False,False,False
307971,57.214898,-3.823997,2,60,2022,2,18.0,809,1009,False,...,False,False,False,False,False,True,False,False,True,False


In [ ]:
y

,Accident_Severity_Encoded
0,1
1,1
2,0
3,1
4,1
...,...
307968,0
307969,0
307970,0
307971,1


In [ ]:
y.shape

(307973,)

In [ ]:
df_encoded.shape

(307973, 57)

In [ ]:
df['Accident_Severity'].value_counts()

,count
Accident_Severity,
Slight,263280
Serious,40740
Fatal,3953


In [ ]:
X.columns

Index(['Latitude', 'Longitude', 'Number_of_Vehicles', 'Speed_limit',
       'Accident_Year', 'Accident_Month', 'Accident_Hour', 'District_Freq',
       'Police_Force_Freq', 'Day_of_Week_Monday', 'Day_of_Week_Saturday',
       'Day_of_Week_Sunday', 'Day_of_Week_Thursday', 'Day_of_Week_Tuesday',
       'Day_of_Week_Wednesday', 'Road_Type_One way street',
       'Road_Type_Roundabout', 'Road_Type_Single carriageway',
       'Road_Type_Slip road', 'Road_Surface_Conditions_Flood over 3cm. deep',
       'Road_Surface_Conditions_Frost or ice', 'Road_Surface_Conditions_Snow',
       'Road_Surface_Conditions_Wet or damp',
       'Weather_Conditions_Fine no high winds',
       'Weather_Conditions_Fog or mist', 'Weather_Conditions_Other',
       'Weather_Conditions_Raining + high winds',
       'Weather_Conditions_Raining no high winds',
       'Weather_Conditions_Snowing + high winds',
       'Weather_Conditions_Snowing no high winds',
       'Light_Conditions_Darkness - lights lit',
       'Lig

In [31]:
# -------------------------------------------------------------------
# STEP 1: Train-Test Split (First & Most Important)
# -------------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [32]:
print(f"Training features shape: {X_train.shape}")
print(f"Testing features shape: {X_test.shape}")

Training features shape: (246378, 52)
Testing features shape: (61595, 52)


In [33]:
# -------------------------------------------------------------------
# STEP 2: Scale Continuous Features
# -------------------------------------------------------------------
# Scale only numerical continuous columns to avoid affecting binary 1/0 flags
num_cols = [
    'Latitude', 'Longitude', 'Number_of_Vehicles', 'Speed_limit',
    'Accident_Year', 'Accident_Month', 'Accident_Hour',
    'District_Freq', 'Police_Force_Freq'
]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])  # Transform test using train stats

In [34]:
# -------------------------------------------------------------------
# STEP 3: Balance the Training Data using SMOTE
# -------------------------------------------------------------------
print("\nBefore SMOTE (Train Set):")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE (Train Set):")
print(y_train_resampled.value_counts())


Before SMOTE (Train Set):
Accident_Severity_Encoded
0    210624
1     32592
2      3162
Name: count, dtype: int64

After SMOTE (Train Set):
Accident_Severity_Encoded
1    210624
0    210624
2    210624
Name: count, dtype: int64


In [ ]:
# -------------------------------------------------------------------
# STEP 4: Train XGBoost Model
# -------------------------------------------------------------------
model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    tree_method='hist'  # Optimized for speed on large datasets
)

model.fit(X_train_resampled, y_train_resampled)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# -------------------------------------------------------------------
# STEP 5: Evaluate on Unseen Test Set
# -------------------------------------------------------------------
y_pred = model.predict(X_test)

print("\n--- Model Performance ---")
print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Slight', 'Serious', 'Fatal']))


--- Model Performance ---
Accuracy Score: 0.853218605406283

Classification Report:
               precision    recall  f1-score   support

      Slight       0.86      1.00      0.92     52656
     Serious       0.33      0.01      0.02      8148
       Fatal       0.30      0.06      0.09       791

    accuracy                           0.85     61595
   macro avg       0.49      0.35      0.34     61595
weighted avg       0.78      0.85      0.79     61595



In [ ]:
df.columns

Index(['Day_of_Week', 'Junction_Control', 'Junction_Detail',
       'Accident_Severity', 'Latitude', 'Light_Conditions',
       'Local_Authority_(District)', 'Longitude', 'Number_of_Vehicles',
       'Police_Force', 'Road_Surface_Conditions', 'Road_Type', 'Speed_limit',
       'Urban_or_Rural_Area', 'Weather_Conditions', 'Vehicle_Type',
       'Accident_Year', 'Accident_Month', 'Accident_Hour',
       'Accident_Severity_Encoded', 'Vehicle_Type_Grouped', 'District_Freq',
       'Police_Force_Freq'],
      dtype='object')

In [ ]:
# passing sample weights directly to xgboost
from sklearn.utils.class_weight import compute_sample_weight

# Calculate weights inversely proportional to class frequencies
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    tree_method='hist'
)

# Pass sample weights directly to fit (WITHOUT SMOTE)
model.fit(X_train, y_train, sample_weight=sample_weights)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=300,
              n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
# 1. Generate predictions on the untouched test set
y_pred = model.predict(X_test)

# 2. Print overall accuracy
acc = accuracy_score(y_test, y_pred)
print(f"Overall Accuracy: {acc:.4f}\n")

# 3. Print the detailed classification report (Precision, Recall, F1-Score)
print("--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=['Slight', 'Serious', 'Fatal']))

# 4. Print the Confusion Matrix to see raw correct vs incorrect counts
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

Overall Accuracy: 0.5626

--- Classification Report ---
              precision    recall  f1-score   support

      Slight       0.91      0.59      0.72     52656
     Serious       0.19      0.38      0.26      8148
       Fatal       0.04      0.56      0.07       791

    accuracy                           0.56     61595
   macro avg       0.38      0.51      0.35     61595
weighted avg       0.80      0.56      0.65     61595

--- Confusion Matrix ---
[[31098 12770  8788]
 [ 2969  3115  2064]
 [  144   206   441]]


In [ ]:
# Predict probabilities for each class
y_proba = model.predict_proba(X_test)

# Assign prediction based on custom thresholds
custom_preds = []
for p in y_proba:
    if p[2] > 0.20:      # Flag as 'Fatal' if probability > 20%
        custom_preds.append(2)
    elif p[1] > 0.30:    # Flag as 'Serious' if probability > 30%
        custom_preds.append(1)
    else:
        custom_preds.append(0)

print("--- Performance with Custom Thresholds ---")
print(classification_report(y_test, custom_preds, target_names=['Slight', 'Serious', 'Fatal']))

--- Performance with Custom Thresholds ---
              precision    recall  f1-score   support

      Slight       0.93      0.19      0.32     52656
     Serious       0.12      0.35      0.18      8148
       Fatal       0.02      0.80      0.05       791

    accuracy                           0.22     61595
   macro avg       0.36      0.45      0.18     61595
weighted avg       0.81      0.22      0.29     61595



In [ ]:
import numpy as np
import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from xgboost import XGBClassifier

# -------------------------------------------------------------------
# STEP 1: Compute Mild/Dampened Class Weights
# -------------------------------------------------------------------
classes = np.unique(y_train)
# Calculate standard balanced weights
raw_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

# Dampen weights using square root to avoid over-penalizing 'Slight' accidents
dampened_weights = np.sqrt(raw_weights)
class_weight_dict = dict(zip(classes, dampened_weights))

# Map dampened weights to every sample in y_train
sample_weights_mild = np.array([class_weight_dict[y] for y in y_train])

# -------------------------------------------------------------------
# STEP 2: Train XGBoost Model
# -------------------------------------------------------------------
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42,
    tree_method='hist'
)

model.fit(X_train, y_train, sample_weight=sample_weights_mild)

# -------------------------------------------------------------------
# STEP 3: Generate Probability Predictions & Apply Custom Thresholds
# -------------------------------------------------------------------
# Get predicted probabilities for each class [P(Slight), P(Serious), P(Fatal)]
y_proba = model.predict_proba(X_test)

# Apply custom decision thresholds to catch severe cases without wrecking accuracy
custom_preds = []
for p in y_proba:
    if p[2] > 0.15:       # Flag as 'Fatal' if probability > 15%
        custom_preds.append(2)
    elif p[1] > 0.28:     # Flag as 'Serious' if probability > 28%
        custom_preds.append(1)
    else:
        custom_preds.append(0)  # Default to 'Slight'

# -------------------------------------------------------------------
# STEP 4: Evaluate Performance & Calculate Accuracy
# -------------------------------------------------------------------
acc = accuracy_score(y_test, custom_preds)

print("=" * 50)
print(f" FINAL OVERALL ACCURACY: {acc:.4f} ({acc * 100:.2f}%)")
print("=" * 50)

print("\n--- Classification Report ---")
print(classification_report(y_test, custom_preds, target_names=['Slight', 'Serious', 'Fatal']))

print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, custom_preds))

 FINAL OVERALL ACCURACY: 0.6188 (61.88%)

--- Classification Report ---
              precision    recall  f1-score   support

      Slight       0.90      0.66      0.76     52656
     Serious       0.20      0.40      0.26      8148
       Fatal       0.05      0.45      0.09       791

    accuracy                           0.62     61595
   macro avg       0.38      0.50      0.37     61595
weighted avg       0.80      0.62      0.69     61595

--- Confusion Matrix ---
[[34490 13190  4976]
 [ 3481  3271  1396]
 [  201   233   357]]


# USE THIS PREDICTION(1)

**TRYING ANOTHER METHOD!!!**

In [35]:
# Step 1 — Prepare the features and target

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

# Remove helper columns created by the previous experiments.
# We will encode the original columns afresh.
data = df.drop(
    columns=[
        "Accident_Severity_Encoded",
        "Vehicle_Type_Grouped",
        "District_Freq",
        "Police_Force_Freq"
    ],
    errors="ignore"
).copy()

# Separate input features and target
X = data.drop(columns=["Accident_Severity"])
y = data["Accident_Severity"].map({
    "Slight": 0,
    "Serious": 1,
    "Fatal": 2
})

# Check the target before training
print("Target distribution:")
print(y.value_counts().sort_index())

# Split BEFORE fitting encoders or imputers
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Identify columns by their data types
numeric_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

print("\nNumerical columns:", numeric_cols)
print("\nCategorical columns:", categorical_cols)


Target distribution:
Accident_Severity
0    263280
1     40740
2      3953
Name: count, dtype: int64

Numerical columns: ['Latitude', 'Longitude', 'Number_of_Vehicles', 'Speed_limit', 'Accident_Year', 'Accident_Month', 'Accident_Hour']

Categorical columns: ['Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Light_Conditions', 'Local_Authority_(District)', 'Police_Force', 'Road_Surface_Conditions', 'Road_Type', 'Urban_or_Rural_Area', 'Weather_Conditions', 'Vehicle_Type']


In [36]:
# Step 2 — Encode categories correctly

# Numerical preprocessing
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical preprocessing
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combine the two pipelines
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])


In [37]:
# Step 3 — Train XGBoost with moderate class weights

# Calculate class weights using training labels only
balanced_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

# Reduce the strength of the weights
sample_weights = np.sqrt(balanced_weights)

# Build one complete model pipeline
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        num_class=3,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=5,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1
    ))
])

# Train without SMOTE
model.fit(
    X_train,
    y_train,
    classifier__sample_weight=sample_weights
)

print("Training completed!")


Training completed!


In [38]:
# Step 4 — Evaluate the model

y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(
    "Balanced Accuracy:",
    round(balanced_accuracy_score(y_test, y_pred), 4)
)
print(
    "Macro F1:",
    round(f1_score(y_test, y_pred, average="macro"), 4)
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Slight", "Serious", "Fatal"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))


Accuracy: 0.845
Balanced Accuracy: 0.3726
Macro F1: 0.3774

Classification Report:
              precision    recall  f1-score   support

      Slight       0.86      0.98      0.92     52656
     Serious       0.31      0.06      0.10      8148
       Fatal       0.22      0.08      0.12       791

    accuracy                           0.85     61595
   macro avg       0.46      0.37      0.38     61595
weighted avg       0.78      0.85      0.80     61595


Confusion Matrix:
[[51520   955   181]
 [ 7631   465    52]
 [  663    63    65]]


In [39]:
# saving the model and dumping it!!
import joblib

# Save the trained model in .joblib format
joblib.dump(model, "accident_severity_model.joblib")

print("Model saved successfully!")


Model saved successfully!


## USE THIS ONE FOR PREDICTION(3)

In [ ]:
# Get the fitted preprocessor from our pipeline
preprocessor = model.named_steps["preprocessor"]

# Transform the training data into encoded numerical data
X_train_encoded = preprocessor.transform(X_train)

print(X_train_encoded.shape)

(246378, 536)


In [ ]:
# # Get all feature names after preprocessing
# encoded_columns = preprocessor.get_feature_names_out()

# print(encoded_columns)

In [ ]:
X_train.columns

Index(['Day_of_Week', 'Junction_Control', 'Junction_Detail', 'Latitude',
       'Light_Conditions', 'Local_Authority_(District)', 'Longitude',
       'Number_of_Vehicles', 'Police_Force', 'Road_Surface_Conditions',
       'Road_Type', 'Speed_limit', 'Urban_or_Rural_Area', 'Weather_Conditions',
       'Vehicle_Type', 'Accident_Year', 'Accident_Month', 'Accident_Hour'],
      dtype='object')

In [ ]:
y_train

,Accident_Severity
116604,1
272094,0
267916,0
137445,0
33957,0
...,...
200481,0
168760,0
250531,1
32687,0


In [ ]:
# CATBOOST ALGORITHM

In [ ]:
%pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
# using smotnec

In [ ]:

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

# Remove helper columns from earlier experiments
data = df.drop(
    columns=[
        "Accident_Severity_Encoded",
        "Vehicle_Type_Grouped",
        "District_Freq",
        "Police_Force_Freq"
    ],
    errors="ignore"
).copy()

# Clean the target labels
data["Accident_Severity"] = (
    data["Accident_Severity"]
    .astype("string")
    .str.strip()
    .str.title()
    .replace({"Fetal": "Fatal"})
)

data = data.dropna(subset=["Accident_Severity"])

data = data[
    data["Accident_Severity"].isin(
        ["Slight", "Serious", "Fatal"]
    )
].copy()

X = data.drop(columns=["Accident_Severity"])

y = data["Accident_Severity"].map({
    "Slight": 0,
    "Serious": 1,
    "Fatal": 2
}).astype("int32")

# 60% training, 20% validation, 20% testing
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev,
    test_size=0.25,
    random_state=42,
    stratify=y_dev
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts().sort_index())


Training shape: (184783, 18)
Validation shape: (61595, 18)
Testing shape: (61595, 18)

Training target distribution:
Accident_Severity
0    157968
1     24444
2      2371
Name: count, dtype: int64


In [ ]:
# Step 2 — Impute missing values and prepare SMOTENC

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from imblearn.over_sampling import SMOTENC

numeric_cols = X_train.select_dtypes(
    include="number"
).columns.tolist()

categorical_cols = X_train.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()

# Impute using training data only
num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

X_train_s = X_train.copy()
X_val_s = X_val.copy()
X_test_s = X_test.copy()

if numeric_cols:
    X_train_s[numeric_cols] = num_imputer.fit_transform(
        X_train[numeric_cols]
    )
    X_val_s[numeric_cols] = num_imputer.transform(
        X_val[numeric_cols]
    )
    X_test_s[numeric_cols] = num_imputer.transform(
        X_test[numeric_cols]
    )

if categorical_cols:
    X_train_s[categorical_cols] = cat_imputer.fit_transform(
        X_train[categorical_cols]
    )
    X_val_s[categorical_cols] = cat_imputer.transform(
        X_val[categorical_cols]
    )
    X_test_s[categorical_cols] = cat_imputer.transform(
        X_test[categorical_cols]
    )

# Map categorical values to integer codes.
# Unknown validation/test categories receive -1.
category_maps = {}

for col in categorical_cols:
    categories = pd.Index(
        X_train_s[col].astype(str).unique()
    )

    category_maps[col] = {
        value: idx for idx, value in enumerate(categories)
    }

    for frame in (X_train_s, X_val_s, X_test_s):
        frame[col] = (
            frame[col].astype(str)
            .map(category_maps[col])
            .fillna(-1)
            .astype("int32")
        )

categorical_indices = [
    X_train_s.columns.get_loc(col)
    for col in categorical_cols
]

print("Categorical features:", len(categorical_cols))
print("Numerical features:", len(numeric_cols))


Categorical features: 11
Numerical features: 7


In [ ]:
# Step 3 — Apply SMOTENC

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    sampling_strategy={
        1: 80000,
        2: 20000
    },
    random_state=42,
    k_neighbors=3
)

X_train_bal, y_train_bal = smotenc.fit_resample(
    X_train_s,
    y_train
)

print("Before resampling:")
print(y_train.value_counts().sort_index())

print("\nAfter SMOTENC:")
print(pd.Series(y_train_bal).value_counts().sort_index())


Before resampling:
Accident_Severity
0    157968
1     24444
2      2371
Name: count, dtype: int64

After SMOTENC:
Accident_Severity
0    157968
1     80000
2     20000
Name: count, dtype: int64


In [ ]:
# Step 4 — Train XGBoost on the resampled data

from xgboost import XGBClassifier
import time

preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", "passthrough", numeric_cols),
        ("categorical", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=True,
            dtype=np.float32
        ), categorical_cols)
    ],
    sparse_threshold=1.0
)

X_train_enc = preprocessor.fit_transform(X_train_bal)
X_val_enc = preprocessor.transform(X_val_s)
X_test_enc = preprocessor.transform(X_test_s)

model_smotenc = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    tree_method="hist",
    eval_metric="mlogloss",
    early_stopping_rounds=40,
    random_state=42,
    n_jobs=4
)

start = time.time()

model_smotenc.fit(
    X_train_enc,
    y_train_bal,
    eval_set=[(X_val_enc, y_val)],
    verbose=50
)

print(
    "Training time:",
    round((time.time() - start) / 60, 2),
    "minutes"
)


[0]	validation_0-mlogloss:0.60306
[50]	validation_0-mlogloss:0.51395
[100]	validation_0-mlogloss:0.49545
[150]	validation_0-mlogloss:0.48642
[200]	validation_0-mlogloss:0.48040
[250]	validation_0-mlogloss:0.47585
[300]	validation_0-mlogloss:0.47236
[350]	validation_0-mlogloss:0.46905
[399]	validation_0-mlogloss:0.46615
Training time: 6.37 minutes


In [ ]:
# Step 5 — Evaluate the model

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

y_pred = model_smotenc.predict(X_test_enc)

print("Accuracy:",
      round(accuracy_score(y_test, y_pred), 4))

print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_test, y_pred), 4))

print("Macro F1:",
      round(f1_score(y_test, y_pred, average="macro"), 4))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    labels=[0, 1, 2],
    target_names=["Slight", "Serious", "Fatal"],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred, labels=[0, 1, 2]))


Accuracy: 0.8538
Balanced Accuracy: 0.3495
Macro F1: 0.3374

Classification Report:
              precision    recall  f1-score   support

      Slight       0.86      1.00      0.92     52656
     Serious       0.32      0.01      0.01      8148
       Fatal       0.33      0.05      0.08       791

    accuracy                           0.85     61595
   macro avg       0.50      0.35      0.34     61595
weighted avg       0.78      0.85      0.79     61595


Confusion Matrix:
[[52510    89    57]
 [ 8085    46    17]
 [  746     9    36]]
